# ERP003950 (мышь) — симуляция merged reads через InSilicoSeq — вариант 250bp

Тренирует свою KDE error-модель на реальных reads этого ERP003950 run
(`iss model`, встроенный сабкоманд `insilicoseq`), затем гоняет
`iss generate --sequence_type amplicon` этой моделью — чтобы получить
**250bp** reads, как в реальном датасете (встроенная модель `miseq` даёт
301bp, см. `simulate_mouse_merged_insilicoseq.ipynb`).

`templates.fasta` / `read_counts.tsv` **переиспользуются** из 301bp-прогона
(построены в его шаге 3, от модели не зависят — пересчитывать дедуп
млн merged reads заново незачем). Модель и fastq-выход этого прогона —
в своей отдельной папке, рядом с 301bp.

**Требует `bowtie2` + `samtools` в `bcr_env`** (их там по умолчанию нет):

```bash
conda install -c bioconda -c conda-forge bowtie2 samtools -y
```

Kernel: **BCR Pipeline** (`bcr_env`), OneQ task
`bbc68913-4463-433d-b647-c3b8a76555ba`.


### 1. Env check

In [ ]:
import os, sys, sysconfig, subprocess

_ENV_CANDIDATES = [
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
]
_CONDA_ENV = next((p for p in _ENV_CANDIDATES if os.path.isdir(p + "/bin")), _ENV_CANDIDATES[-1])
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [
    _CONDA_ENV + "/lib/python3.11/site-packages",
    _CONDA_ENV + "/lib/python3.12/site-packages",
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)

print(f"Using env: {_CONDA_ENV}")
for tool in ("iss", "bowtie2", "bowtie2-build", "samtools"):
    path = subprocess.run(["which", tool], capture_output=True, text=True).stdout.strip()
    if not path:
        raise RuntimeError(
            f"'{tool}' not found in PATH. Install with:\n"
            f"  conda install -c bioconda -c conda-forge bowtie2 samtools -y  # (iss should already be present)"
        )
    print(tool, "->", path)
print(subprocess.run(["iss", "--version"], capture_output=True, text=True).stderr.strip())


### 2. Config

In [ ]:
from pathlib import Path

VOLUME = Path("/data/user/epishkin")
DATASET = "ERP003950"
SAMPLES = ["ERR346596", "ERR346597", "ERR346598", "ERR346599", "ERR346600", "ERR346601"]

MERGED_FASTQ_DIR = VOLUME / "results" / DATASET / "merged" / "fastq"
RAW_FASTQ_DIR = VOLUME / "raw" / DATASET  # ERR346596_1.fastq.gz / _2.fastq.gz

# templates.fasta / read_counts.tsv общие с 301bp-прогоном (от модели не зависят, пересчитывать дорого)
SHARED_TEMPLATES_DIR = VOLUME / "results" / DATASET / "simulated" / "insilicoseq" / "templates"

# собственное дерево выхода этого прогона, рядом с 301bp (не внутри него)
OUT_BASE = VOLUME / "results" / DATASET / "simulated" / "insilicoseq_250bp"
MODEL_DIR = OUT_BASE / "model"
FASTQ_DIR = OUT_BASE / "fastq"
LOGS_DIR = OUT_BASE / "logs"
QC_DIR = OUT_BASE / "qc"
for d in (MODEL_DIR, FASTQ_DIR, LOGS_DIR, QC_DIR):
    d.mkdir(parents=True, exist_ok=True)

REF_FASTA = MODEL_DIR / "ERP003950_all_samples_reference.fasta"
BOWTIE2_INDEX = MODEL_DIR / "ERP003950_bt2_index"
BAM_PATH = MODEL_DIR / "ERP003950_real_reads_vs_merged.bam"
CUSTOM_MODEL_PREFIX = MODEL_DIR / "ERP003950_mouse_miseq250"
CUSTOM_MODEL_NPZ = Path(str(CUSTOM_MODEL_PREFIX) + ".npz")

# --- ручки ---
SEQUENCE_TYPE = "amplicon"
NPROC = 8
SEED = 42
COMPRESS = True
FORCE = False


### 3. Train the 250bp error model

`iss model` сам не выравнивает — нужен готовый BAM реальных reads на
референс. Референс — уже собранные (`AssemblePairs.py`) merged sequences
этого же датасета: выравниваем raw R1/R2 обратно на них через `bowtie2`.
Тяжёлый шаг (~4.6M read pairs по всем 6 samples) — heartbeat каждые 30с.


In [ ]:
import time

def run_with_heartbeat(cmd, log_path, heartbeat=30, shell=False):
    t0 = time.time()
    with open(log_path, "w") as log_h:
        proc = subprocess.Popen(cmd, stdout=log_h, stderr=subprocess.STDOUT, text=True, shell=shell)
        label = cmd if shell else " ".join(cmd)
        print(f"[run] {label}\n  pid={proc.pid} log={log_path}")
        while proc.poll() is None:
            print(f"  still running: pid={proc.pid} elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)
    elapsed = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"command failed (exit {proc.returncode}); see {log_path}")
    print(f"done: elapsed={elapsed/60:.1f} min")

def iter_fastq(path):
    import gzip
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                return
            seq = h.readline().rstrip("\n")
            h.readline()  # +
            h.readline()  # качество
            yield seq


In [ ]:
# 3a. Собрать один reference FASTA из merged (assemble-pass) reads всех samples
def merged_fastq_to_fasta(sample, out_handle):
    fq = MERGED_FASTQ_DIR / f"{sample}_assemble-pass.fastq.gz"
    n = 0
    for i, seq in enumerate(iter_fastq(fq)):
        out_handle.write(f">{sample}_{i}\n{seq}\n")
        n += 1
    return n

if REF_FASTA.exists() and not FORCE:
    print(f"[skip] {REF_FASTA} exists")
else:
    total = 0
    with open(REF_FASTA, "w") as out_h:
        for sample in SAMPLES:
            n = merged_fastq_to_fasta(sample, out_h)
            total += n
            print(f"  {sample}: {n:,} reference sequences")
    print(f"wrote {REF_FASTA} ({total:,} sequences total)")


In [ ]:
# 3b. bowtie2-build
index_done_marker = Path(str(BOWTIE2_INDEX) + ".1.bt2")
if index_done_marker.exists() and not FORCE:
    print(f"[skip] bowtie2 index already built: {BOWTIE2_INDEX}")
else:
    run_with_heartbeat(
        ["bowtie2-build", "--threads", str(NPROC), str(REF_FASTA), str(BOWTIE2_INDEX)],
        LOGS_DIR / "bowtie2_build.log",
    )


In [ ]:
# 3c. Выровнять raw R1/R2 (все 6 samples) на merged-read референс -> отсортированный+индексированный BAM
if BAM_PATH.exists() and not FORCE:
    print(f"[skip] {BAM_PATH} exists")
else:
    r1_list = ",".join(str(RAW_FASTQ_DIR / f"{s}_1.fastq.gz") for s in SAMPLES)
    r2_list = ",".join(str(RAW_FASTQ_DIR / f"{s}_2.fastq.gz") for s in SAMPLES)
    align_cmd = (
        f"bowtie2 --local -p {NPROC} -x {BOWTIE2_INDEX} -1 {r1_list} -2 {r2_list} "
        f"2> {LOGS_DIR / 'bowtie2_align.log'} "
        f"| samtools view -bS - "
        f"| samtools sort -@ {NPROC} -o {BAM_PATH} -"
    )
    run_with_heartbeat(align_cmd, LOGS_DIR / "bowtie2_align_pipe.log", shell=True)
    subprocess.run(["samtools", "index", str(BAM_PATH)], check=True)
    print(f"indexed {BAM_PATH}")


In [ ]:
# 3d. iss model: построить кастомную KDE error-модель из BAM
if CUSTOM_MODEL_NPZ.exists() and not FORCE:
    print(f"[skip] {CUSTOM_MODEL_NPZ} exists")
else:
    run_with_heartbeat(
        ["iss", "model", "-b", str(BAM_PATH), "-o", str(CUSTOM_MODEL_PREFIX)],
        LOGS_DIR / "iss_model.log",
    )

from iss.error_models.kde import KDErrorModel
em = KDErrorModel(str(CUSTOM_MODEL_NPZ))
print(f"model: {CUSTOM_MODEL_NPZ}")
print(f"read_length = {em.read_length}")
if em.read_length != 250:
    print(f"WARNING: expected read_length=250, model actually has {em.read_length}")


### 4. Run `iss generate` per sample (250bp model)

Использует шаблоны, уже построенные в шаге 3 `simulate_mouse_merged_insilicoseq.ipynb`
(`SHARED_TEMPLATES_DIR`) — если их там нет, сначала прогони тот ноутбук до
конца шага 3.


In [ ]:
for sample in SAMPLES:
    fa = SHARED_TEMPLATES_DIR / f"{sample}_templates.fasta"
    tsv = SHARED_TEMPLATES_DIR / f"{sample}_read_counts.tsv"
    if not (fa.exists() and tsv.exists()):
        raise FileNotFoundError(
            f"Missing shared templates for {sample}: {fa} / {tsv}\n"
            "Run step 3 in simulate_mouse_merged_insilicoseq.ipynb first."
        )
print("templates OK for all samples")


In [ ]:
import csv

def run_iss_generate(sample, model=str(CUSTOM_MODEL_NPZ), sequence_type=SEQUENCE_TYPE, nproc=NPROC,
                      seed=SEED, compress=COMPRESS, force=FORCE):
    templates_fa = SHARED_TEMPLATES_DIR / f"{sample}_templates.fasta"
    counts_tsv = SHARED_TEMPLATES_DIR / f"{sample}_read_counts.tsv"
    out_prefix = FASTQ_DIR / sample
    ext = ".fastq.gz" if compress else ".fastq"
    r1_out = Path(str(out_prefix) + f"_R1{ext}")
    r2_out = Path(str(out_prefix) + f"_R2{ext}")

    if r1_out.exists() and r2_out.exists() and not force:
        print(f"[{sample}] [skip] {r1_out.name} exists")
        return {"sample": sample, "status": "skipped"}

    stdout_path = LOGS_DIR / f"{sample}_iss.stdout.txt"
    stderr_path = LOGS_DIR / f"{sample}_iss.stderr.txt"

    cmd = [
        "iss", "generate",
        "--genomes", str(templates_fa),
        "--readcount_file", str(counts_tsv),
        "--sequence_type", sequence_type,
        "--model", model,
        "--cpus", str(nproc),
        "--output", str(out_prefix),
    ]
    if seed is not None:
        cmd += ["--seed", str(seed)]
    if compress:
        cmd.append("--compress")

    print(f"[{sample}] [run] {' '.join(cmd)}")
    t0 = time.time()
    with open(stdout_path, "w") as out_h, open(stderr_path, "w") as err_h:
        proc = subprocess.Popen(cmd, stdout=out_h, stderr=err_h, text=True)
        print(f"  pid={proc.pid} stdout={stdout_path.name} stderr={stderr_path.name}")
        while True:
            rc = proc.poll()
            if rc is not None:
                break
            print(f"  still running: pid={proc.pid} elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(30)
    elapsed = time.time() - t0
    if rc != 0:
        raise RuntimeError(f"iss generate failed for {sample} (exit {rc}); see {stderr_path}")

    n_r1 = sum(1 for _ in iter_fastq(r1_out))
    print(f"[{sample}] done: R1={n_r1:,} reads elapsed={elapsed/60:.1f} min")
    return {
        "sample": sample, "status": "done", "elapsed_sec": f"{elapsed:.1f}",
        "reads_generated": n_r1, "r1_fastq": str(r1_out), "r2_fastq": str(r2_out),
    }


In [ ]:
sim_rows = [run_iss_generate(sample) for sample in SAMPLES]

qc_path = QC_DIR / "simulate_qc.tsv"
done_rows = [r for r in sim_rows if r.get("status") == "done"]
if done_rows:
    with open(qc_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(done_rows[0].keys()), delimiter="\t")
        writer.writeheader()
        writer.writerows(done_rows)
    print(f"wrote {qc_path}")

total_generated = sum(r.get("reads_generated", 0) for r in sim_rows if r.get("status") == "done")
print(f"total simulated read pairs: {total_generated:,}")


### Notes

- Модель и BAM/index — в `results/ERP003950/simulated/insilicoseq_250bp/model/`.
- Fastq-выход — `.../insilicoseq_250bp/fastq/{sample}_R1/_R2.fastq.gz`, рядом
  с 301bp прогоном (`.../simulated/insilicoseq/fastq/`), в отдельной папке.
- `templates/` и ground truth (`read_counts.tsv`) общие с 301bp прогоном —
  один и тот же набор шаблонов, разные модели.
